In [0]:
from pyspark.sql.functions import col, current_timestamp, to_json, struct, lit, trim, upper, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

vendor_name = "georesults"
layer_name = "silver_conform"

print(f"--- Starting Silver Conformance for: {vendor_name} ---")

# 1. Read directly from the Bronze table
df_bronze = spark.table("inlap.bronze.georesults")

# State mapping dictionary for full name to 2-letter code resolution
state_mapping = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "FLORIDA": "FL", "GEORGIA": "GA",
    "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA",
    "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD",
    "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS", "MISSOURI": "MO",
    "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV", "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM", "NEW YORK": "NY", "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH",
    "OKLAHOMA": "OK", "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT",
    "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY"
}

# 2. Field casting, normalization, and state translation
df_cleaned = df_bronze \
    .withColumn("lat", col("geo_lat").cast("double")) \
    .withColumn("lon", col("geo_long").cast("double")) \
    .withColumn("confidence_score", col("confidence_score").cast("double")) \
    .withColumn("region_upper", upper(trim(col("region")))) \
    .withColumn("site_id", col("record_id").cast("string"))

# Map full state names to 2-letter codes using a `when` chain
state_expr = lit(None)
for full_name, code in state_mapping.items():
    state_expr = when(col("region_upper") == full_name, lit(code)).otherwise(state_expr)

# Fallback: if it's already a 2-letter code or unrecognized
df_mapped = df_cleaned \
    .withColumn("state_code", when(col("region_upper").rlike("^[A-Z]{2}$"), col("region_upper")).otherwise(state_expr))

# 3. Define Quality Rules
# - Latitude must be between -90 and 90 (catches 999.999)
# - Longitude must be between -180 and 180
# - State code must be successfully mapped (not null)
# - Confidence score must meet a minimum threshold (e.g., >= 0.5)
# - site_id must not be null
valid_condition = (
    col("lat").isNotNull() & 
    col("lon").isNotNull() & 
    (col("lat").between(-90.0, 90.0)) & 
    (col("lon").between(-180.0, 180.0)) &
    (col("state_code").isNotNull()) &
    (col("confidence_score").isNotNull()) &
    (col("confidence_score") >= 0.5) &
    (col("site_id").isNotNull())
)

# 4. Split into Valid vs Quarantined
df_passed_dq = df_mapped.filter(valid_condition)
df_dq_quarantine = df_mapped.filter(~valid_condition) \
    .withColumn(
        "failure_reason", 
        lit("unrecognized region, coordinate out of bounds, missing site_id, or low confidence score")
    )

# 5. Handle duplicates among valid records (keep highest confidence or latest per site_id)
window_spec = Window.partitionBy("site_id").orderBy(col("confidence_score").desc())
df_with_rn = df_passed_dq.withColumn("row_num", row_number().over(window_spec))

df_valid = df_with_rn.filter(col("row_num") == 1).drop("row_num", "region_upper", "geo_lat", "geo_long", "region")
df_dup_quarantine = df_with_rn.filter(col("row_num") > 1) \
    .drop("row_num") \
    .withColumn("failure_reason", lit("duplicate site_id record"))

# 6. Combine all quarantine records and structure final output
df_all_quarantine = df_dq_quarantine.unionByName(df_dup_quarantine)
df_quarantine_final = df_all_quarantine \
    .withColumn("source_vendor", lit(vendor_name)) \
    .withColumn("quarantine_timestamp", current_timestamp()) \
    .withColumn("raw_record", to_json(struct([col(c) for c in df_bronze.columns]))) \
    .select("raw_record", "failure_reason", "source_vendor", "quarantine_timestamp")

# Safely lowercase column names for the conformed table (overwrite region with state_code)
df_valid = df_valid \
    .withColumn("state", col("state_code")) \
    .drop("state_code")

df_valid = df_valid.select([col(c).alias(c.lower()) for c in df_valid.columns])

# 7. Write to Silver Conformed and Quarantine tables
df_valid.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("inlap.silver.georesults_conformed")

df_quarantine_final.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("inlap.silver.quarantine_records")

# 8. Log Audit Metrics
rows_read = df_bronze.count()
rows_passed = df_valid.count()
rows_quarantined = df_quarantine_final.count()

spark.sql(f"""
    INSERT INTO inlap.control.audit_log 
    VALUES (
        '{vendor_name}', 
        '{layer_name}', 
        current_timestamp(), 
        'SUCCESS', 
        {rows_read}, 
        {rows_passed}, 
        {rows_quarantined}
    )
""")

print(f"Conformance complete for {vendor_name}. Read: {rows_read} | Passed: {rows_passed} | Quarantined: {rows_quarantined}")